<a href="https://colab.research.google.com/github/Grazipolachini/MBA/blob/main/ELT_RISCO_DE_CREDITO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Parte 1 - Ingestão de dados na camada Raw**

In [1]:
#Instala Pyspark

!pip install pyspark

In [2]:
#Inicia motor de processamento distribuido

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EtlRiscoDecredito") \
    .getOrCreate()

spark

In [4]:
#Monta drike para ler arquivos
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
#Le arquivos
df_app = spark.read.csv(
    "/content/drive/MyDrive/EtlRiscoDeCredito/data/Raw/application_train.csv",
    header=True,
    inferSchema=True
)
df_bureau = spark.read.csv(
    "/content/drive/MyDrive/EtlRiscoDeCredito/data/Raw/bureau.csv",
    header=True,
    inferSchema=True
)

In [6]:
#Valida subida dos arquvivos
df_app.printSchema()
df_app.show(5)
df_app.count()

root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- TARGET: integer (nullable = true)
 |-- NAME_CONTRACT_TYPE: string (nullable = true)
 |-- CODE_GENDER: string (nullable = true)
 |-- FLAG_OWN_CAR: string (nullable = true)
 |-- FLAG_OWN_REALTY: string (nullable = true)
 |-- CNT_CHILDREN: integer (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- NAME_TYPE_SUITE: string (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- NAME_FAMILY_STATUS: string (nullable = true)
 |-- NAME_HOUSING_TYPE: string (nullable = true)
 |-- REGION_POPULATION_RELATIVE: double (nullable = true)
 |-- DAYS_BIRTH: integer (nullable = true)
 |-- DAYS_EMPLOYED: integer (nullable = true)
 |-- DAYS_REGISTRATION: double (nullable = true)
 |-- DAYS_ID_PUBLISH: integer (nullable = true)
 |-- OWN_CAR_AG

307511

In [7]:
#valida subida dos arquivos
df_bureau.printSchema()
df_bureau.show(5)
df_bureau.count()

root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- SK_ID_BUREAU: integer (nullable = true)
 |-- CREDIT_ACTIVE: string (nullable = true)
 |-- CREDIT_CURRENCY: string (nullable = true)
 |-- DAYS_CREDIT: integer (nullable = true)
 |-- CREDIT_DAY_OVERDUE: integer (nullable = true)
 |-- DAYS_CREDIT_ENDDATE: double (nullable = true)
 |-- DAYS_ENDDATE_FACT: double (nullable = true)
 |-- AMT_CREDIT_MAX_OVERDUE: double (nullable = true)
 |-- CNT_CREDIT_PROLONG: integer (nullable = true)
 |-- AMT_CREDIT_SUM: double (nullable = true)
 |-- AMT_CREDIT_SUM_DEBT: double (nullable = true)
 |-- AMT_CREDIT_SUM_LIMIT: double (nullable = true)
 |-- AMT_CREDIT_SUM_OVERDUE: double (nullable = true)
 |-- CREDIT_TYPE: string (nullable = true)
 |-- DAYS_CREDIT_UPDATE: integer (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)

+----------+------------+-------------+---------------+-----------+------------------+-------------------+-----------------+----------------------+------------------+--------

1716428

In [8]:
#Salva como arquivo parquet
df_app.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/EtlRiscoDeCredito/data/raw_parquet/application_train")

df_bureau.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/EtlRiscoDeCredito/data/raw_parquet/bureau"
)

**Parte 2 - Tratamento na camada Trusted**

In [9]:
#Verifica volume das tabelas
print("Linhas:", df_app.count())
print("Colunas:", len(df_app.columns))
print("Linhas:", df_bureau.count())
print("Colunas:", len(df_bureau.columns))

Linhas: 307511
Colunas: 122
Linhas: 1716428
Colunas: 17


In [10]:
#Valida distribuição da base
df_app.groupBy("TARGET").count().show()
#Valida duplicidade na chave
df_app.select("SK_ID_CURR").distinct().count()
df_app.count()

+------+------+
|TARGET| count|
+------+------+
|     1| 24825|
|     0|282686|
+------+------+



307511

In [11]:
#Validação de nulos na base

from pyspark.sql.functions import col, count, when

total = df_app.count()

null_df = df_app.select([
    (count(when(col(c).isNull(), c)) / total).alias(c)
    for c in df_app.columns
])

null_df.show()

+----------+------+------------------+-----------+------------+---------------+------------+----------------+----------+--------------------+--------------------+--------------------+----------------+-------------------+------------------+-----------------+--------------------------+----------+-------------+-----------------+---------------+------------------+----------+--------------+---------------+----------------+----------+----------+-------------------+--------------------+--------------------+---------------------------+--------------------------+-----------------------+--------------------------+--------------------------+---------------------------+----------------------+----------------------+-----------------------+-----------------+------------------+--------------------+-------------------+------------------+------------------+---------------------------+------------------+------------------+-----------------+-----------------+------------------+------------------+--------

In [12]:
from pyspark.sql.functions import col, when, sum as spark_sum

#Valida os nulos somente nas colunas que desejamos materializar
cols_interesse = [
    "SK_ID_CURR",
    "TARGET",
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3"
]

total = df_app.count()

null_df = df_app.select([
    (spark_sum(when(col(c).isNull(), 1).otherwise(0)) / total).alias(c)
    for c in cols_interesse
])

null_df.show()

+----------+------+------------------+-----------+----------------+----------+--------------------+--------------------+------------+--------------------+----------+-------------+------------------+--------------------+-------------------+
|SK_ID_CURR|TARGET|NAME_CONTRACT_TYPE|CODE_GENDER|AMT_INCOME_TOTAL|AMT_CREDIT|         AMT_ANNUITY|     AMT_GOODS_PRICE|CNT_CHILDREN|     CNT_FAM_MEMBERS|DAYS_BIRTH|DAYS_EMPLOYED|      EXT_SOURCE_1|        EXT_SOURCE_2|       EXT_SOURCE_3|
+----------+------+------------------+-----------+----------------+----------+--------------------+--------------------+------------+--------------------+----------+-------------+------------------+--------------------+-------------------+
|       0.0|   0.0|               0.0|        0.0|             0.0|       0.0|3.902299429939091...|9.040327012692228E-4|         0.0|6.503832383231819E-6|       0.0|          0.0|0.5638107254699832|0.002146264686466...|0.19825307062186392|
+----------+------+------------------+--

In [13]:

#Preenche com 0 somente colunas numericas nulas
cols_numericas = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3"
]

df_app = df_app.fillna(0, subset=cols_numericas)
#valida
df_app.select(cols_numericas).summary("count").show()
df_app.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in cols_numericas
]).show()

+-------+----------------+----------+-----------+---------------+------------+---------------+----------+-------------+------------+------------+------------+
|summary|AMT_INCOME_TOTAL|AMT_CREDIT|AMT_ANNUITY|AMT_GOODS_PRICE|CNT_CHILDREN|CNT_FAM_MEMBERS|DAYS_BIRTH|DAYS_EMPLOYED|EXT_SOURCE_1|EXT_SOURCE_2|EXT_SOURCE_3|
+-------+----------------+----------+-----------+---------------+------------+---------------+----------+-------------+------------+------------+------------+
|  count|          307511|    307511|     307511|         307511|      307511|         307511|    307511|       307511|      307511|      307511|      307511|
+-------+----------------+----------+-----------+---------------+------------+---------------+----------+-------------+------------+------------+------------+

+----------------+----------+-----------+---------------+------------+---------------+----------+-------------+------------+------------+------------+
|AMT_INCOME_TOTAL|AMT_CREDIT|AMT_ANNUITY|AMT_GOODS_PR

In [14]:
#Preenche com ND somente colunas numericas nulas
cols_categoricas = [
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER"
]
#valida
df_app = df_app.fillna(0, subset=cols_categoricas)
#valida
df_app.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in cols_categoricas
]).show()

+------------------+-----------+
|NAME_CONTRACT_TYPE|CODE_GENDER|
+------------------+-----------+
|                 0|          0|
+------------------+-----------+



In [15]:
#Salva na tabela trusted somente as colunas desejadas e tratadas e salva em parquet

cols_final = [
    "SK_ID_CURR",
    "TARGET",
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3"
]

df_final = df_app.select(cols_final)

#salva em parquet

df_final.write.mode("overwrite").parquet("/content/drive/MyDrive/EtlRiscoDeCredito/data/trusted_parquet/application_train")

In [ ]:
#Valida os nulos somente nas colunas que desejamos materializar na tabela bureau
cols_interesse_bureau = [
    "SK_ID_CURR",
    "SK_ID_BUREAU",
    "CREDIT_ACTIVE",
    "AMT_CREDIT_SUM",
    "AMT_CREDIT_SUM_DEBT",
    "CREDIT_DAY_OVERDUE"
]

total = df_bureau.count()

null_df = df_bureau.select([
    (spark_sum(when(col(c).isNull(), 1).otherwise(0)) / total).alias(c)
    for c in cols_interesse_bureau
])

null_df.show()

+----------+------------+-------------+--------------------+-------------------+------------------+
|SK_ID_CURR|SK_ID_BUREAU|CREDIT_ACTIVE|      AMT_CREDIT_SUM|AMT_CREDIT_SUM_DEBT|CREDIT_DAY_OVERDUE|
+----------+------------+-------------+--------------------+-------------------+------------------+
|       0.0|         0.0|          0.0|7.573868522303295E-6|0.15011931755948982|               0.0|
+----------+------------+-------------+--------------------+-------------------+------------------+



In [16]:

#Preenche com 0 somente colunas numericas nulas
cols_numericas_bureau = [
    "SK_ID_CURR",
    "SK_ID_BUREAU",
    "CREDIT_ACTIVE",
    "AMT_CREDIT_SUM",
    "AMT_CREDIT_SUM_DEBT",
    "CREDIT_DAY_OVERDUE",
    "DAYS_CREDIT"
]

df_bureau = df_bureau.fillna(0, subset=cols_numericas_bureau)
#valida
df_bureau.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in cols_numericas_bureau
]).show()

df_bureau.show(5)

+----------+------------+-------------+--------------+-------------------+------------------+-----------+
|SK_ID_CURR|SK_ID_BUREAU|CREDIT_ACTIVE|AMT_CREDIT_SUM|AMT_CREDIT_SUM_DEBT|CREDIT_DAY_OVERDUE|DAYS_CREDIT|
+----------+------------+-------------+--------------+-------------------+------------------+-----------+
|         0|           0|            0|             0|                  0|                 0|          0|
+----------+------------+-------------+--------------+-------------------+------------------+-----------+

+----------+------------+-------------+---------------+-----------+------------------+-------------------+-----------------+----------------------+------------------+--------------+-------------------+--------------------+----------------------+---------------+------------------+-----------+
|SK_ID_CURR|SK_ID_BUREAU|CREDIT_ACTIVE|CREDIT_CURRENCY|DAYS_CREDIT|CREDIT_DAY_OVERDUE|DAYS_CREDIT_ENDDATE|DAYS_ENDDATE_FACT|AMT_CREDIT_MAX_OVERDUE|CNT_CREDIT_PROLONG|AMT_CREDIT

In [17]:
#seleção de colunas necessárias para materialização da camada gold bureau
cols_bureau_trusted = [
    "SK_ID_CURR",
    "SK_ID_BUREAU",
    "CREDIT_ACTIVE",
    "AMT_CREDIT_SUM",
    "AMT_CREDIT_SUM_DEBT",
    "CREDIT_DAY_OVERDUE",
    "DAYS_CREDIT"
]

df_bureau_trusted = df_bureau.select(*cols_bureau_trusted)

#Materializa trusted bureau em parquet
df_bureau_trusted.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/EtlRiscoDeCredito/data/trusted_parquet/bureau"
)

In [46]:
df_bureau_trusted.show(10)

+----------+------------+-------------+--------------+-------------------+------------------+-----------+
|SK_ID_CURR|SK_ID_BUREAU|CREDIT_ACTIVE|AMT_CREDIT_SUM|AMT_CREDIT_SUM_DEBT|CREDIT_DAY_OVERDUE|DAYS_CREDIT|
+----------+------------+-------------+--------------+-------------------+------------------+-----------+
|    215354|     5714462|       Closed|       91323.0|                0.0|                 0|       -497|
|    215354|     5714463|       Active|      225000.0|           171342.0|                 0|       -208|
|    215354|     5714464|       Active|      464323.5|                0.0|                 0|       -203|
|    215354|     5714465|       Active|       90000.0|                0.0|                 0|       -203|
|    215354|     5714466|       Active|     2700000.0|                0.0|                 0|       -629|
|    215354|     5714467|       Active|      180000.0|           71017.38|                 0|       -273|
|    215354|     5714468|       Active|       

In [ ]:
Parte 3 - Disponibilização de tabela na camada gold

In [18]:
#Para inciciar o cruzamento para a camada bronze, vamos usar como premissa, considerar a última linha de registro de bureau do cliente

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("SK_ID_CURR").orderBy(col("DAYS_CREDIT").desc(),
                                                       col("CREDIT_ACTIVE").desc())

bureau_last = df_bureau_trusted.withColumn(
    "rn",
    row_number().over(window_spec)
).filter(
    col("rn") == 1
).drop("rn")

In [19]:
#Valida se existe duplicidade
bureau_last.groupBy("SK_ID_CURR").count().filter("count > 1").show()

+----------+-----+
|SK_ID_CURR|count|
+----------+-----+
+----------+-----+



In [20]:
#join entre as tabelas utilizando brodcast para otimizar processamento

from pyspark.sql.functions import broadcast

df_gold = df_final.join(
    broadcast(bureau_last),
    on="SK_ID_CURR",
    how="left"
).fillna({
    "SK_ID_BUREAU": 0,
    "CREDIT_ACTIVE": "ND",
    "AMT_CREDIT_SUM": 0,
    "AMT_CREDIT_SUM_DEBT": 0,
    "CREDIT_DAY_OVERDUE": 0,
    "DAYS_CREDIT": 0
})

#valida

print("Linhas application:", df_final.count())
print("Linhas gold:", df_gold.count())

df_gold.show(5)

Linhas application: 307511
Linhas gold: 307511
+----------+------+------------------+-----------+----------------+----------+-----------+---------------+------------+---------------+----------+-------------+-------------------+------------------+-------------------+------------+-------------+--------------+-------------------+------------------+-----------+
|SK_ID_CURR|TARGET|NAME_CONTRACT_TYPE|CODE_GENDER|AMT_INCOME_TOTAL|AMT_CREDIT|AMT_ANNUITY|AMT_GOODS_PRICE|CNT_CHILDREN|CNT_FAM_MEMBERS|DAYS_BIRTH|DAYS_EMPLOYED|       EXT_SOURCE_1|      EXT_SOURCE_2|       EXT_SOURCE_3|SK_ID_BUREAU|CREDIT_ACTIVE|AMT_CREDIT_SUM|AMT_CREDIT_SUM_DEBT|CREDIT_DAY_OVERDUE|DAYS_CREDIT|
+----------+------+------------------+-----------+----------------+----------+-----------+---------------+------------+---------------+----------+-------------+-------------------+------------------+-------------------+------------+-------------+--------------+-------------------+------------------+-----------+
|    100002|  

In [21]:
#Materializa tabela gold em Parquet

df_gold.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/EtlRiscoDeCredito/data/gold_parquet/gold_risco_de_credito"
)

In [22]:
#Valida antes de exportar

df_gold.count()
df_gold.show(5)
df_gold.printSchema()

+----------+------+------------------+-----------+----------------+----------+-----------+---------------+------------+---------------+----------+-------------+-------------------+------------------+-------------------+------------+-------------+--------------+-------------------+------------------+-----------+
|SK_ID_CURR|TARGET|NAME_CONTRACT_TYPE|CODE_GENDER|AMT_INCOME_TOTAL|AMT_CREDIT|AMT_ANNUITY|AMT_GOODS_PRICE|CNT_CHILDREN|CNT_FAM_MEMBERS|DAYS_BIRTH|DAYS_EMPLOYED|       EXT_SOURCE_1|      EXT_SOURCE_2|       EXT_SOURCE_3|SK_ID_BUREAU|CREDIT_ACTIVE|AMT_CREDIT_SUM|AMT_CREDIT_SUM_DEBT|CREDIT_DAY_OVERDUE|DAYS_CREDIT|
+----------+------+------------------+-----------+----------------+----------+-----------+---------------+------------+---------------+----------+-------------+-------------------+------------------+-------------------+------------+-------------+--------------+-------------------+------------------+-----------+
|    100002|     1|        Cash loans|          M|        202

In [23]:
#Exporta bronze em csv
df_gold.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "/content/drive/MyDrive/EtlRiscoDeCredito/data/gold_csv/gold_risco_de_credito"
)